# V2.1 — SAFE OPTIMIZATION: Giải thích chi tiết & so sánh với V1


## V2 tối ưu những gì so với V1?

| Vấn đề ở V1 | Cách V2 khắc phục |
|---|---|
| `make_gaussian_kernel_1d(sigma)` bị gọi lại **mỗi octave × mỗi scale** (4×5 = 20 lần), dù kernel Gaussian ở mỗi `scale` không đổi giữa các octave (σ chỉ phụ thuộc `scale`, không phụ thuộc `octave`) | Precompute `host_kernels`/`radii` **một lần duy nhất** trước cả 2 vòng lặp octave/scale |
| Mỗi scale tự `cuda.device_array(...)` cấp phát mới cho `d_temp`, `d_blurred` → tốn `cudaMalloc`/`cudaFree` liên tục (chi phí driver-level, không phải compute) | Cấp phát `d_temp`, `d_out` **một lần / octave**, tái sử dụng (ghi đè) qua các scale trong octave đó |
| Luôn `copy_to_host()` **mọi** scale ở mọi octave, kể cả khi chỉ cần đo tốc độ (không cần dữ liệu ảnh) | Thêm cờ `return_host`: `True` = verify mode (copy đủ để so MAE), `False` = benchmark mode (chỉ copy đúng 1 ảnh mid-scale/octave để downsample) |
| — | Kernel tính toán (`gaussian_blur_row_kernel`, `gaussian_blur_col_kernel`, `reflect_scipy`) **giữ nguyên y hệt V1** |



## Phần 1: Import thư viện và thiết lập môi trường

Giống hệt V1: `sys/os/time/math/numpy/glob` cho hạ tầng, `numba.cuda` để viết kernel,
`cv2` (OpenCV) bọc trong `try/except` để chạy SIFT tham chiếu và đọc/ghi ảnh, ép
`stdout` sang UTF-8 để terminal Windows hiển thị đúng tiếng Việt có dấu. Không có gì
thay đổi ở phần này so với V1 — vì tối ưu của V2 nằm ở tầng điều phối GPU, không phải
tầng import.

In [1]:
import sys
import os
import time
import math
import numpy as np
import glob

# Ep UTF-8 cho Windows terminal
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

from numba import cuda

# OpenCV chi dung de load anh
try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False


### Mount Google Drive và nạp module V0 (bản CPU tham chiếu)

Giống hệt V1: mount Drive để đọc `V0_1_with_lib.py` (bản CPU dùng `scipy`, dùng làm
chuẩn đối chiếu MAE) và bộ ảnh DIV2K. `sys.path` được dọn (loại path cũ nếu có) rồi
chèn `folder_path` lên đầu để đảm bảo import đúng bản module mới nhất, tránh dùng
cache cũ. Nếu import lỗi, `V0_AVAILABLE = False` để các bước verify/benchmark có thể
bị bỏ qua an toàn thay vì crash toàn bộ chương trình.

In [2]:
import importlib.util
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

folder_path = '/content/drive/MyDrive/Colab Notebooks'
sys.path = [p for p in sys.path if p != folder_path]
sys.path.insert(0, folder_path)

try:
    from V0_1_with_lib import (
        build_gaussian_pyramid_with_lib,
        compute_dog_pyramid,
        create_synthetic_div2k_image,
        load_single_image,
        get_div2k_image_paths
    )
    V0_AVAILABLE = True
except ImportError:
    V0_AVAILABLE = False
    print("[ERROR] Khong tim thay V0_1_with_lib.py - bo qua tinh nang benchmark.")


Mounted at /content/drive


## Phần 2: CUDA kernels — GIỮ NGUYÊN từ V1 (đã pass MAE)

Đây là điểm mấu chốt của tư duy "SAFE OPTIMIZATION": `reflect_scipy` (xử lý biên kiểu
`scipy.ndimage` reflect) và 2 kernel `gaussian_blur_row_kernel` /
`gaussian_blur_col_kernel` (separable convolution 1D theo hàng rồi theo cột) được
**copy y hệt** logic đã kiểm chứng ở V1 (chỉ đổi tên hàm reflect từ `reflect_idx` →
`reflect_scipy`, nội dung giống nhau). Không có branch, không có shared memory, không
có gì mới ở tầng tính toán per-pixel — mỗi thread vẫn tương ứng 1 pixel output và đọc
trực tiếp từ Global Memory như V1.

**Vì sao giữ nguyên mà không viết lại kernel tốt hơn ngay?** Vì mục tiêu của V2 là một
bước tối ưu **an toàn, có kiểm soát**: đổi phần điều phối (Python/host) trước, giữ
nguyên phần lõi CUDA đã pass test — để nếu tốc độ tăng lên, ta biết chắc chắn đó là do
giảm overhead host chứ không lẫn với sai số/hiệu ứng lạ do đổi thuật toán kernel.

In [3]:
# Giữ kernel V1 đã pass MAE (reflect + separable)
@cuda.jit(device=True, inline=True)
def reflect_scipy(i, n):
    if n <= 1:
        return 0
    p = 2 * n
    i = i % p
    if i < 0:
        i += p
    if i >= n:
        i = p - 1 - i
    return i

@cuda.jit
def gaussian_blur_row_kernel(d_input, d_output, d_kernel, radius):
    x, y = cuda.grid(2)
    H, W = d_input.shape
    if x >= W or y >= H:
        return

    acc = 0.0
    for k in range(-radius, radius + 1):
        xx = reflect_scipy(x + k, W)
        acc += d_input[y, xx] * d_kernel[k + radius]
    d_output[y, x] = acc

@cuda.jit
def gaussian_blur_col_kernel(d_input, d_output, d_kernel, radius):
    x, y = cuda.grid(2)
    H, W = d_input.shape
    if x >= W or y >= H:
        return

    acc = 0.0
    for k in range(-radius, radius + 1):
        yy = reflect_scipy(y + k, H)
        acc += d_input[yy, x] * d_kernel[k + radius]
    d_output[y, x] = acc


**Lưu ý nhỏ về `reflect_scipy` so với `reflect_idx` (V1):** V1 dùng vòng `while`
lặp cho đến khi chỉ số hợp lệ. V2 dùng công thức modulo (`i % p` với `p = 2*n`) để quy
chỉ số về một chu kỳ phản chiếu duy nhất trong **một bước tính toán**, không cần lặp
nhiều vòng. Về mặt toán học hai cách cho cùng kết quả (đã được verify lại bằng MAE
trong `verify_correctness` bên dưới), nhưng bản modulo giúp giảm phân kỳ nhánh
(branch divergence) một chút cho các trường hợp `radius` lớn hơn nhiều lần `n` — tuy
vậy đây vẫn là tối ưu vi mô, không phải trọng tâm của V2.

### Hàm `make_gaussian_kernel_1d(sigma, truncate=4.0)`

Giống hệt V1: sinh kernel Gaussian 1D chuẩn hoá theo đúng công thức `scipy` dùng
(`radius = int(truncate*sigma + 0.5)`), để đảm bảo kích thước kernel và giá trị khớp
với CPU reference khi verify MAE.

In [4]:
def make_gaussian_kernel_1d(sigma, truncate=4.0):
    radius = int(truncate * float(sigma) + 0.5)
    x = np.arange(-radius, radius + 1, dtype=np.float32)
    k = np.exp(-(x * x) / (2.0 * sigma * sigma))
    k /= k.sum()
    return k.astype(np.float32)


## Phần 3: `build_gaussian_pyramid_gpu_v2` — nơi các tối ưu thực sự nằm ở đây

Đây là hàm điều phối chính, và cũng là toàn bộ sự khác biệt về hiệu năng giữa V1 và
V2. So sánh từng khối với V1:

**1. Precompute kernel Gaussian MỘT LẦN cho toàn bộ pyramid (không phải mỗi octave)**

```python
host_kernels = []
radii = []
for s in range(num_scales):
    sigma = sigma_base * (k ** s)
    hk = make_gaussian_kernel_1d(sigma, truncate=4.0)
    host_kernels.append(hk)
    radii.append(hk.shape[0] // 2)
```

Ở V1, đoạn code này nằm **bên trong** vòng lặp `for octave: for scale:`, tức bị gọi
lại `num_octaves × num_scales = 20` lần dù kết quả của `make_gaussian_kernel_1d(sigma)`
chỉ phụ thuộc `sigma`, mà `sigma = sigma_base * (k**s)` chỉ phụ thuộc `scale` — **không
đổi giữa các octave**. V2 tính đúng `num_scales = 5` lần, ở ngoài mọi vòng lặp. Đây là
optimization thuần CPU (loại bỏ tính toán NumPy dư thừa), không liên quan CUDA.

**2. Cấp phát buffer MỘT LẦN mỗi octave, tái sử dụng qua các scale**

```python
d_temp = cuda.device_array((H, W), dtype=np.float32)
d_out  = cuda.device_array((H, W), dtype=np.float32)
```

nằm ngoài vòng lặp `for s in range(num_scales)`. Ở V1, mỗi scale tự cấp phát
`d_temp`/`d_blurred` mới bằng `cuda.device_array(...)`. Với `H, W` không đổi trong
suốt 1 octave, việc gọi `cudaMalloc` 5 lần/octave (thay vì 1 lần) chỉ để ghi đè cùng
kích thước bộ nhớ là chi phí driver-level thuần tuý, không mang lại giá trị gì — CUDA
memory allocation có latency đáng kể (thường vài chục µs) so với thời gian một kernel
convolution nhỏ, nên với ảnh vừa/nhỏ chi phí này có thể chiếm tỷ trọng lớn trong tổng
thời gian.

**3. Cache device kernel theo octave**

```python
dev_kernels = [cuda.to_device(hk) for hk in host_kernels]
```

Việc `cuda.to_device()` (copy Host→Device) vẫn lặp lại mỗi octave ở V2 (vì object GPU
buffer không tự nhiên sống bền qua các octave trong thiết kế hiện tại), nhưng **so
với V1 nó đã giảm từ 20 lần `to_device` xuống còn `num_octaves = 4` lần** thay vì gắn
liền với việc tính lại kernel mỗi lần — đây là điểm còn có thể tối ưu tiếp (đưa hẳn ra
ngoài vòng lặp octave), nhưng V2 dừng ở mức "an toàn, dễ kiểm chứng" thay vì tối ưu
triệt để ngay.

**4. Cờ `return_host`: tách "verify mode" khỏi "benchmark mode"**

```python
if return_host:
    out_host = d_out.copy_to_host()
    octave_host.append(out_host)
    ...
else:
    if s == mid_idx:
        d_mid = cuda.device_array_like(d_out)
        d_mid.copy_to_device(d_out)
```

Đây là tối ưu quan trọng nhất về mặt số lượng lệnh Device→Host. V1 luôn
`copy_to_host()` mọi ảnh ở mọi scale (`num_octaves × num_scales = 20` lần copy toàn bộ
ảnh), kể cả khi mục đích chỉ là đo tốc độ và không ai đọc dữ liệu ảnh đó. Copy
Device→Host qua PCIe là một trong những thao tác **chậm nhất** trong toàn bộ pipeline
(chậm hơn nhiều so với một kernel convolution nhỏ), vì vậy giảm được 19/20 lần copy
không cần thiết trong benchmark mode có tác động rất lớn đến thời gian đo được — lớn
hơn nhiều so với việc tối ưu bản thân kernel convolution.


In [5]:
def build_gaussian_pyramid_gpu_v2(
    image,
    num_octaves=4,
    num_scales=5,
    sigma_base=1.6,
    return_host=True,          # True cho verify; False cho benchmark nhanh
):
    """
    V2 optimized (exactness-preserving):
    - Giữ kernel row/col V1 đã pass MAE
    - Cache d_kernel theo octave shape
    - Reuse buffer trong octave
    - Optional: benchmark mode khong copy host tung scale
    """
    pyramid = []
    current_img = np.ascontiguousarray(image.astype(np.float32))
    d_current = cuda.to_device(current_img)

    k = 2.0 ** (1.0 / num_scales)

    # precompute host kernels 1 lần
    host_kernels = []
    radii = []
    for s in range(num_scales):
        sigma = sigma_base * (k ** s)
        hk = make_gaussian_kernel_1d(sigma, truncate=4.0)
        host_kernels.append(hk)
        radii.append(hk.shape[0] // 2)

    # chọn block "vừa phải" để cân bằng
    block = (16, 16)

    for _ in range(num_octaves):
        H, W = d_current.shape
        grid = (math.ceil(W / block[0]), math.ceil(H / block[1]))

        # cache device kernels cho octave hiện tại (1 lần/octave)
        dev_kernels = [cuda.to_device(hk) for hk in host_kernels]

        # reuse buffers
        d_temp = cuda.device_array((H, W), dtype=np.float32)
        d_out  = cuda.device_array((H, W), dtype=np.float32)

        if return_host:
            octave_host = []

        d_mid = None
        mid_idx = num_scales // 2

        for s in range(num_scales):
            d_kernel = dev_kernels[s]
            radius = radii[s]

            gaussian_blur_row_kernel[grid, block](d_current, d_temp, d_kernel, radius)
            gaussian_blur_col_kernel[grid, block](d_temp, d_out, d_kernel, radius)

            if return_host:
                out_host = d_out.copy_to_host()
                octave_host.append(out_host)
                if s == mid_idx:
                    # downsample dùng host (verify mode)
                    mid_host = out_host
            else:
                if s == mid_idx:
                    # benchmark mode: chỉ giữ d_mid device để downsample nhanh
                    d_mid = cuda.device_array_like(d_out)
                    d_mid.copy_to_device(d_out)

        if return_host:
            pyramid.append(octave_host)
            current_img = np.ascontiguousarray(mid_host[::2, ::2])
            d_current = cuda.to_device(current_img)
        else:
            # benchmark mode: copy đúng 1 ảnh mid mỗi octave
            mid_host = d_mid.copy_to_host()
            current_img = np.ascontiguousarray(mid_host[::2, ::2])
            d_current = cuda.to_device(current_img)

    return pyramid if return_host else None


## Vì sao V2 chưa dùng Shared Memory?

Có 2 lý
do cụ thể vì sao V2 chưa làm điều đó:


**1. Bán kính kernel (`radius`) thay đổi theo từng `scale`, gây khó khăn kỹ thuật thật
sự cho Shared Memory trong Numba CUDA.** Shared memory trên GPU (`cuda.shared.array(shape, dtype)`)
đòi hỏi `shape` là **hằng số biết trước lúc biên dịch** (compile-time constant), không
thể truyền `radius` (biến runtime, khác nhau ở mỗi trong 5 scale) làm kích thước tile.
Muốn dùng Shared Memory đúng cách cho pyramid này, cần một trong hai hướng:
- Cấp phát tile theo `radius` lớn nhất trong toàn bộ pyramid → lãng phí shared memory
  (và giảm occupancy — số block chạy đồng thời trên 1 SM) cho các scale có `radius`
  nhỏ hơn nhiều.
- Sinh kernel CUDA riêng cho từng `radius` (specialize theo compile-time constant) →
  tăng đáng kể độ phức tạp code và thời gian JIT-compile (Numba phải compile lại mỗi
  kernel mới), đi ngược tinh thần "SAFE" của bước tối ưu này.

**2. Halo overhead của Shared Memory tăng theo `radius`, làm giảm lợi ích đúng ở những
scale tốn thời gian nhất.** Với block 16×16 và `radius` càng lớn (scale sau, `sigma`
càng lớn), vùng "halo" (viền dữ liệu thừa cần nạp thêm quanh mỗi tile để phục vụ các
thread ở biên tile) chiếm tỷ trọng ngày càng lớn so với vùng lõi 16×16 hữu ích — có
những trường hợp `radius` xấp xỉ hoặc vượt quá kích thước block, khiến lợi ích của
Shared Memory (giảm đọc trùng lặp) bị bào mòn bởi chính chi phí nạp halo.


**Kết luận:** Shared Memory là bước tối ưu hợp lý cho V3 (hoặc bản kế tiếp), khi đã
tách riêng kernel theo `radius` cố định (ví dụ specialize theo từng octave/scale) hoặc
chấp nhận đánh đổi giữa padding lãng phí và độ phức tạp code, đồng thời có bộ verify
MAE sẵn sàng để kiểm chứng lại từ đầu. Ở V2, việc tối ưu tầng host (giảm `cudaMalloc`,
giảm copy Device→Host, giảm tính lại kernel Gaussian dư thừa) mang lại tỷ lệ
lợi-ích/rủi-ro tốt hơn nhiều cho một bước tối ưu "an toàn".

## Phần 4: Verify + Benchmark

`verify_correctness` xây pyramid bằng cả CPU (`scipy`, qua `build_gaussian_pyramid_with_lib`)
và GPU V2 (`return_host=True` để lấy đủ dữ liệu so sánh), rồi tính MAE trung bình
(`np.mean(np.abs(ref - out))`, khác V1 dùng `.max()`) cho từng cặp `(octave, scale)`,
in cảnh báo nếu vượt ngưỡng `1e-3`. Việc verify pass ở đây khẳng định các tối ưu host
(cache kernel, reuse buffer, tách benchmark mode) **không làm thay đổi kết quả số học**
so với V1 — đúng như kỳ vọng, vì kernel tính toán không đổi.

In [6]:
def verify_correctness(img, num_octaves, num_scales, sigma_base=1.6):
    """Kiem tra do chinh xac (MAE) giua GPU V2 va CPU scipy reference."""
    print("\n[VERIFY] Kiem tra do chinh xac MAE (GPU V2 vs CPU scipy)...")

    ref_pyramid = build_gaussian_pyramid_with_lib(img, num_octaves, num_scales, sigma_base)
    gpu_pyramid = build_gaussian_pyramid_gpu_v2(img, num_octaves, num_scales, sigma_base)

    max_mae = 0.0
    passed = True

    for o in range(num_octaves):
        for s in range(num_scales):
            ref = ref_pyramid[o][s]
            out = gpu_pyramid[o][s]

            mae = float(np.mean(np.abs(ref - out)))
            max_mae = max(max_mae, mae)

            if mae > 1e-3:
                print(f"  [ERROR] Octave {o}, Scale {s}: MAE = {mae:.5e} > 1e-3")
                passed = False

    if passed:
        print(f"  [PASS] Tat ca layers co MAE < 1e-3 (Max MAE = {max_mae:.5e})")
    else:
        print(f"  [FAIL] Do chinh xac khong dat yeu cau (Max MAE = {max_mae:.5e})")


## Phần 5: `main()` — điều phối toàn bộ chương trình



In [7]:
def run_sift_opencv(gray: np.ndarray, max_detect_dim=1200):
    h, w = gray.shape
    scale = 1.0
    gray_detect = gray

    if max_detect_dim is not None and max(h, w) > max_detect_dim:
        scale = max_detect_dim / max(h, w)
        gray_detect = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    t0 = time.time()
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(gray_detect, None)
    elapsed = time.time() - t0

    if scale != 1.0:
        inv_scale = 1.0 / scale
        kp = [
            cv2.KeyPoint(x=k.pt[0] * inv_scale, y=k.pt[1] * inv_scale, size=k.size * inv_scale,
                         angle=k.angle, response=k.response, octave=k.octave, class_id=k.class_id)
            for k in kp
        ]
    return kp, des, elapsed, scale


def main():
    print("=" * 70)
    print("  V2_gpu_optimized.py -- GPU V2 (Optimized Numba CUDA)")
    print("=" * 70)

    NUM_OCTAVES = 4
    NUM_SCALES = 5
    N_RUNS = 3
    SIGMA_BASE = 1.6

    OUTPUT_DIR = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not V0_AVAILABLE:
        print("Khong co thu vien/reference V0 de verify benchmark. Ket thuc.")
        return

    if not CV2_AVAILABLE:
        print("OpenCV chua duoc cai dat -> khong doc duoc DIV2K images.")
        return

    # Verify tren anh synthetic
    synth_img = create_synthetic_div2k_image(seed=42)
    verify_correctness(synth_img, NUM_OCTAVES, NUM_SCALES, SIGMA_BASE)

    # Doc anh DIV2K tu Drive (giong format V1 ban muon)
    div2k_dir = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'DIV2K_train_HR')
    image_paths = sorted(glob.glob(os.path.join(div2k_dir, "*.png")))[:10]

    print("\nDIV2K dir:", div2k_dir)
    print("Found images:", len(image_paths))

    # Warm-up JIT
    dummy = np.zeros((64, 64), dtype=np.float32)
    _ = build_gaussian_pyramid_gpu_v2(
        dummy,
        num_octaves=1,
        num_scales=NUM_SCALES,
        sigma_base=SIGMA_BASE, return_host=False
    )
    cuda.synchronize()

    all_gpu_times = []
    all_cpu_times = []

    if not image_paths:
        print("Khong co anh PNG nao trong DIV2K_train_HR.")
    else:
        print(f"Benchmarking {len(image_paths)} images...")

    for i, img_path in enumerate(image_paths, 1):
        img_color = cv2.imread(img_path)
        if img_color is None:
            print(f"[WARN] Khong doc duoc: {img_path}")
            continue

        gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
        image = gray.astype(np.float32) / 255.0  # khop CPU ref

        # GPU V2
        gpu_times = []
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            _ = build_gaussian_pyramid_gpu_v2(
                image,
                num_octaves=NUM_OCTAVES,
                num_scales=NUM_SCALES,
                sigma_base=SIGMA_BASE
            )
            cuda.synchronize()
            gpu_times.append(time.perf_counter() - t0)
        avg_gpu = 1000 * np.mean(gpu_times)
        all_gpu_times.append(avg_gpu)

        # CPU reference
        cpu_times = []
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            _ = build_gaussian_pyramid_with_lib(
                image,
                num_octaves=NUM_OCTAVES,
                num_scales=NUM_SCALES,
                sigma_base=SIGMA_BASE
            )
            cpu_times.append(time.perf_counter() - t0)
        avg_cpu = 1000 * np.mean(cpu_times)
        all_cpu_times.append(avg_cpu)

        h, w = gray.shape
        print(
            f"[{i}/{len(image_paths)}] {os.path.basename(img_path)} ({w}x{h}) "
            f"| CPU: {avg_cpu:.1f} ms | GPU V2: {avg_gpu:.1f} ms | Speedup: {avg_cpu / avg_gpu:.2f}x"
        )

        # Chạy OpenCV SIFT trên ảnh để lưu kết quả keypoint
        kp, des, t_sift, scale = run_sift_opencv(gray, max_detect_dim=1200)
        name = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(OUTPUT_DIR, f"{name}_v2_result.png")
        img_kp = cv2.drawKeypoints(img_color, kp, None, color=(0, 255, 255), flags=cv2.DRAW_MATCHES_FLAGS_DEFAULT)
        cv2.imwrite(out_path, img_kp)

    if all_gpu_times:
        overall_cpu = float(np.mean(all_cpu_times))
        overall_gpu = float(np.mean(all_gpu_times))
        print("-" * 60)
        print(f"Overall CPU:   {overall_cpu:.1f} ms/image")
        print(f"Overall GPUV2: {overall_gpu:.1f} ms/image")
        print(f"Speedup:       {overall_cpu / overall_gpu:.2f}x")

    print("\n[DONE] V2_gpu_optimized.py hoan thanh.")


if __name__ == "__main__":
    main()


  V2_gpu_optimized.py -- GPU V2 (Optimized Numba CUDA)

[VERIFY] Kiem tra do chinh xac MAE (GPU V2 vs CPU scipy)...
  [PASS] Tat ca layers co MAE < 1e-3 (Max MAE = 1.78703e-07)

DIV2K dir: /content/drive/MyDrive/Colab Notebooks/DIV2K_train_HR
Found images: 10
Benchmarking 10 images...


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[1/10] 0001.png (2040x1404) | CPU: 562.3 ms | GPU V2: 57.9 ms | Speedup: 9.71x
[2/10] 0002.png (2040x1848) | CPU: 764.3 ms | GPU V2: 92.3 ms | Speedup: 8.28x
[3/10] 0003.png (2040x1356) | CPU: 453.9 ms | GPU V2: 73.9 ms | Speedup: 6.14x
[4/10] 0004.png (2040x1344) | CPU: 359.8 ms | GPU V2: 66.3 ms | Speedup: 5.43x
[5/10] 0005.png (1608x2040) | CPU: 940.1 ms | GPU V2: 81.0 ms | Speedup: 11.61x
[6/10] 0006.png (1356x2040) | CPU: 498.5 ms | GPU V2: 70.1 ms | Speedup: 7.11x
[7/10] 0007.png (2040x1356) | CPU: 358.8 ms | GPU V2: 65.2 ms | Speedup: 5.50x
[8/10] 0008.png (2040x1356) | CPU: 374.4 ms | GPU V2: 65.4 ms | Speedup: 5.72x
[9/10] 0009.png (2040x1524) | CPU: 739.1 ms | GPU V2: 80.3 ms | Speedup: 9.21x
[10/10] 0010.png (2040x1644) | CPU: 568.8 ms | GPU V2: 80.4 ms | Speedup: 7.07x
------------------------------------------------------------
Overall CPU:   562.0 ms/image
Overall GPUV2: 73.3 ms/image
Speedup:       7.67x

[DONE] V2_gpu_optimized.py hoan thanh.



**Hướng đi tiếp theo (V3+):**  bước
tối ưu tiếp theo mới nên nhắm vào bên trong kernel — ví dụ dùng Shared Memory với tile
cố định theo octave/scale (chấp nhận biên dịch kernel riêng cho từng `radius`, hoặc cố
định `radius` lớn nhất và chịu lãng phí), hoặc gộp (fuse) 2 lượt row/col convolution
thành 1 kernel để giảm số lần đọc/ghi Global Memory trung gian (`d_temp`).